# YouTube's Recommender System: A Scenario Design Analysis



---

## Introduction

I have always wondered how it is possible that YouTube's system feels like a self-contained echo chamber, capable of alienating the user into a perpetual, individual cybernetic feedback loop of desire and consumption. You sit down to watch one short clip about, say, how sourdough bread is made, and three hours later you are deep in a video about the geopolitics of grain in ancient Mesopotamia, recommended by something that almost looks alive. How does it manage to drag people's attention this way? And how delicate must the balance have to be. Recommend something too weird and the user bounces, too repetitive and they get bored. There is a strange tightrope being walked here, by an algorithm, on behalf of (or perhaps against) us.

This notebook is my attempt to reverse-engineer that experience using Bruce Temkin's Scenario Design framework. I am going to do it twice, once from YouTube's perspective (the organization) and once from the user's perspective, because, as the assignment hints, those two views are not the same and the gap between them is where most of the interesting stuff lives. After that I will piece together what is probably happening under the hood from public sources, and end with some recommendations.

I am not a recommender systems engineer. I am a student who has stared at the YouTube homepage too many times and finally got curious enough to ask: what is this thing actually doing to me?

## Why YouTube?

Out of all the recommender systems I could have picked (Spotify, Netflix, TikTok, Amazon), I chose YouTube because it is the one I feel most strangely about. Spotify recommends music. Amazon recommends things to buy. TikTok is openly, almost honestly, addictive. But YouTube sits in this weird in-between place. It is both informational and entertainment, both a search engine and a consumption funnel. It is where I learn how to fix my bike and where I lose two hours to compilations of cats falling off things.

It also has a long, well-documented history of accusations: the so-called radicalization pipeline, the conspiracy theory rabbit holes, the claim that it nudges people toward more extreme content. Whether or not those claims are fully true (researchers disagree), the fact that they are plausible enough to debate tells you something about the power of the system. With over a billion hours of video watched per day, this is arguably one of the most consequential recommender systems ever built.

## Scenario Design, Take 1: YouTube (the Organization)

Bruce Temkin's framework is three questions. Easy on paper, surprisingly hard to answer honestly. Here is YouTube's view, as best as I can guess it.

### 1) Who are your target users?

YouTube has two types of users that matter inside the product, and a third stakeholder who pays the bills:

- Viewers: around 2.5 billion logged-in monthly users worldwide, spanning every demographic on Earth. The user base is so wide that "target user" almost loses meaning. YouTube does segment them, mostly by watch history, locale, age band, and device.
- Creators: anyone uploading video, from a kid filming Minecraft on a school laptop to MrBeast's industrial-scale studio.
- Advertisers: businesses buying attention. They are not really the "user," but the recommender's success or failure is measured in dollars that come from them.

### 2) What are their key goals?

From YouTube's point of view, the primary goal of the organization is to maximize watch time, and increasingly session time and user satisfaction, because those things correlate with ad impressions, which is the revenue. They also need to keep creators happy enough to keep uploading, and advertisers comfortable enough to keep paying.

### 3) How can you help them accomplish those goals?

YouTube's answer has been: get extremely good at predicting what a given user will click on next, and what they will keep watching once they click. The recommender system is not a feature on YouTube. It is YouTube. Roughly 70% of watch time on the platform comes from algorithmic recommendations, a number that has been cited many times since YouTube's CPO Neal Mohan mentioned it back in 2018.

## Scenario Design, Take 2: The User

Now flipping it. This one is more interesting because user goals and YouTube's goals are not identical. They overlap, but they are not the same thing, and that gap is where most of the controversy lives.

### 1) Who are your target users?

If I am designing for me (and people like me), I am thinking of someone who:

- Comes to YouTube with a specific question or curiosity ("how do I solder wires," "what is causal inference")
- Or comes for entertainment with no specific intent ("I'm bored, show me something")
- Is vaguely aware on some level that the platform is trying to keep them watching, but does not always have the willpower to leave
- Probably trusts YouTube as a source of information more than they should

### 2) What are their key goals?

This is where it gets philosophically slippery. The user has at least three goals, and they often disagree with each other:

- Immediate goal: "Find me a video that satisfies the impulse I just had."
- Session goal: "I have about 30 minutes. Fill it with something worthwhile."
- Long-term goal: "Don't waste my life. Help me become smarter, more skilled, or genuinely entertained, not just dopamine-poked."

YouTube's recommender is excellent at the first goal, decent at the second, and arguably hostile to the third. The user's revealed preference (what they actually click on) and their stated preference (what they would say they want if you asked them on a Sunday morning) often diverge sharply. The algorithm only sees the revealed one.

### 3) How can you help them accomplish those goals?

Honestly, this is where I think there is a real design opportunity that the platform mostly ignores. I save my answer for the Recommendations section.

## Reverse Engineering: What Is YouTube Probably Doing?

I cannot see the source code, but there is actually a fair amount written about the YouTube recommender, both by YouTube engineers and by researchers (and the occasional ex-employee turned critic).

### The 2016 paper (Covington, Adams, Sargin)

The most cited internal description is "Deep Neural Networks for YouTube Recommendations" by Paul Covington, Jay Adams, and Emre Sargin (RecSys 2016). They describe a two-stage funnel:

1. Candidate generation. Narrow YouTube's catalog of billions of videos down to a few hundred candidates. They frame it as an extreme multi-class classification problem: given a user's watch history, predict which video out of millions they will watch next. Done with a deep neural network over embeddings of watched videos, search queries, and demographic features.
2. Ranking. Take the few hundred candidates and score them carefully using a richer feature set: video age, upload time relative to view, click-through rate, and so on. The ranking objective in the paper is expected watch time per impression, not click-through rate, which is interesting because they explicitly note that CTR alone rewards clickbait.

That paper is from 2016. It has almost certainly been replaced or heavily extended since then. I would bet there is a lot more reinforcement learning in the loop now, and very likely transformer-based sequence models on top. But the two-stage architecture (candidate generator plus ranker) seems to still be the basic shape.

As Covington and his coauthors put it, the model has to deal with "scale, freshness, and noise," a phrase that captures what makes YouTube's job different from Amazon's or Netflix's. New videos are uploaded constantly, and most of the implicit signal (watch time, partial views) is noisy.

### What is probably going on now (educated guesses)

From later YouTube blog posts and engineering talks, a few things appear to have been added since:

- Multi-objective optimization. The ranker is not just maximizing watch time anymore. It balances watch time, satisfaction signals (likes, "not interested" clicks, survey responses), and "responsibility" signals, a layer added after the misinformation and radicalization controversies, where so-called borderline content is actively down-ranked.
- Reinforcement learning in the loop. YouTube engineers published "Top-K Off-Policy Correction for a REINFORCE Recommender System" (Chen et al., 2019), explicitly modeling the fact that recommending video A actually changes what the user wants next. That is the cybernetic feedback loop, made formal in math. They are aware of it. They are designing around it.
- Embedding similarity. Every video and every user has a learned vector representation, and similarity in that vector space drives a lot of "you might like" decisions. This is part of why the system feels so coherent and so claustrophobic at the same time.

### The echo chamber question

So why does it feel like a self-contained world? My best guess is that it is a structural consequence of the design, not a deliberate intent:

1. The candidate generator narrows from billions to hundreds based on what is similar to your history.
2. The ranker picks from those hundreds.
3. Whatever you watch becomes new history.
4. Repeat.

This is a positive feedback loop. Even with the diversity tricks the engineers are clearly using (freshness boosts, exploration noise, demoting content too similar to what you just watched), the asymptotic behavior of the loop tends toward a tight cluster. It is not that YouTube wants you in an echo chamber. It is that an echo chamber is what you get when a system is optimized to predict what you will watch, because the easiest prediction is "more of the thing they just watched."

The "delicate balance" feeling is real and worth dwelling on. Recommend something too far from the user's cluster and they bounce; too close and they get bored and bounce a different way. The system is constantly playing this game on a per-user basis, and what you experience as a coherent stream of "things YouTube thinks I want" is in fact a very tight optimization between novelty and familiarity. Tufekci, in her 2018 New York Times op-ed "YouTube, the Great Radicalizer," argued that this balance is biased toward the more extreme, more emotionally activating end of any topic cluster, because emotional activation is what drives watch time. Whether that bias is as strong as she described is debated by later researchers (Hosseinmardi et al., 2021, in PNAS, found weaker effects), but the underlying mechanic, that the loop converges toward emotional engagement, is hard to dispute.

## Recommendations

Here is where I get to be a little ambitious. If I were on the YouTube recommendations team, here is what I would push for. Some of these are small, some of them are big, and a couple of them probably hurt revenue, which is the honest tension at the center of all this.

### 1) Make the long-term-preference signal explicit and editable

Right now the system infers what you want from what you click. That is the cybernetic loop. I would add a layer where users can explicitly tell the system what kind of session they are having ("I'm here to learn," "I'm here to relax," "I want to be challenged") and weight the recommendations accordingly. Some of this exists with topics and channel subscriptions, but it is buried. Make it a first-class control on the homepage.

### 2) An "I just wasted an hour" feedback button

A button, after a session, that lets the user say: "I watched all of this and I regret it." That is a hugely informative signal that no amount of click data captures. It is exactly the gap between revealed and stated preference, and it is currently invisible to the model.

### 3) Genuine exploration, not just adjacent exploration

Periodically inject a recommendation that is deliberately far from your embedding cluster, not random, but pulled from a different valid cluster. Frame it honestly: "Here is something different. We don't think you'd normally click this. Try it?" This breaks the loop without lying about it. The risk, of course, is that watch time goes down. But satisfaction probably goes up, and that is the trade YouTube already claims to want to make.

### 4) Transparency for creators

Creators currently optimize for an opaque signal. Give them a clearer picture of why a video is or is not being recommended, not the full algorithm, but at least the broad categories ("audience overlap with X is low," etc.). This would reduce the clickbait arms race, where everyone makes thumbnails of a screaming face because nobody knows what else to do.

### 5) A "Less of this" control that actually works

The current "Don't recommend channel" and "Not interested" buttons are famously weak. Users have been complaining for years that they barely affect the feed. I would make them much more aggressive in the short term and let users adjust the time horizon ("hide for a week," "hide forever").

### 6) Distinguish search from browse more sharply

When I come to YouTube searching for something specific, I want the recommender to back off and let the search results breathe. When I am idly browsing the homepage, I want the recommender to do its thing. Right now the line is blurry. Even after a search, the sidebar pulls me back into my historical loop, which defeats the purpose of having searched in the first place.

## References

- Covington, P., Adams, J., and Sargin, E. (2016). "Deep Neural Networks for YouTube Recommendations." Proceedings of RecSys '16.
- Chen, M., Beutel, A., Covington, P., Jain, S., Belletti, F., and Chi, E. H. (2019). "Top-K Off-Policy Correction for a REINFORCE Recommender System." WSDM '19.
- Linden, G., Smith, B., and York, J. (2003). "Amazon.com Recommendations: Item-to-Item Collaborative Filtering." IEEE Internet Computing. https://datajobs.com/data-science-repo/Recommender-Systems-[Amazon].pdf
- Spangher, A. (2015). "Building the Next New York Times Recommendation Engine." NYT Open Blog. http://open.blogs.nytimes.com/2015/08/11/building-the-next-new-york-times-recommendation-engine/
- Temkin, B. D. (2004). "Scenario Design: A Disciplined Approach to Customer Experience." Forrester Research.
- Tufekci, Z. (2018). "YouTube, the Great Radicalizer." The New York Times, March 10, 2018.
- Hosseinmardi, H., Ghasemian, A., Clauset, A., Mobius, M., Rothschild, D. M., and Watts, D. J. (2021). "Examining the consumption of radical content on YouTube." PNAS, 118(32).
- Roose, K. (2019). "The Making of a YouTube Radical." The New York Times, June 8, 2019.
- Mohan, N. (2018). Statements on the share of YouTube watch time driven by recommendations, as reported in CNET and elsewhere.